# EinsteinEngine vs. Pure SymPy: Performance Benchmarks

This notebook demonstrates the computational superiority of the **EinsteinEngine** (a hybrid C++/Python tensor engine using `SymEngine`) against the standard pure Python `SymPy` approach. 

We will calculate the 256 components of the **Riemann Curvature Tensor** for the Schwarzschild black hole metric, measuring the execution time of the entire mathematical pipeline (Metric Inversion -> Christoffel Symbols -> Riemann Tensor).

In [1]:

import time
import sympy as sp

# Import our custom high-performance library
from einsteinengine.symbolic.metric import MetricTensor
from einsteinengine.symbolic.riemann import RiemannCurvatureTensor

# 1. Define coordinate symbols
t, r, theta, phi = sp.symbols('t r theta phi', real=True)
M = sp.symbols('M', real=True)
syms = [t, r, theta, phi]
dims = 4

# 2. Define the Schwarzschild Metric (Covariant g_{mu nu})
g_schwarzschild = [
    [-(1 - 2*M/r), 0, 0, 0],
    [0, 1/(1 - 2*M/r), 0, 0],
    [0, 0, r**2, 0],
    [0, 0, 0, r**2 * sp.sin(theta)**2]
]

print("Environment ready and metric defined.")

Environment ready and metric defined.


## Experiment 1: Pure SymPy Execution

In this experiment, we force Python to calculate the entire tensor pipeline using purely `SymPy` and standard nested loops. This represents the traditional, unoptimized approach to symbolic General Relativity.

In [2]:
print("--- Running Pure SymPy Benchmark ---")

start_sp = time.time()

# 1. Initialization and Inversion
g_matrix = sp.Matrix(g_schwarzschild)
g_inv = g_matrix.inv()

# 2. Christoffel Symbols (First & Second kind combined)
Gamma = [[[sp.sympify(0) for _ in range(dims)] for _ in range(dims)] for _ in range(dims)]
for lambda_ in range(dims):
    for mu in range(dims):
        for nu in range(dims):
            suma = 0
            for sigma in range(dims):
                term1 = sp.diff(g_matrix[nu, sigma], syms[mu])
                term2 = sp.diff(g_matrix[mu, sigma], syms[nu])
                term3 = sp.diff(g_matrix[mu, nu], syms[sigma])
                suma += g_inv[lambda_, sigma] * (term1 + term2 - term3)
            Gamma[lambda_][mu][nu] = sp.Rational(1, 2) * suma

# 3. Riemann Curvature Tensor
Riemann = [[[[sp.sympify(0) for _ in range(dims)] for _ in range(dims)] for _ in range(dims)] for _ in range(dims)]
for rho in range(dims):
    for sigma in range(dims):
        for mu in range(dims):
            for nu in range(dims):
                term1 = sp.diff(Gamma[rho][nu][sigma], syms[mu])
                term2 = sp.diff(Gamma[rho][mu][sigma], syms[nu])
                term3 = sum([Gamma[rho][mu][lam] * Gamma[lam][nu][sigma] for lam in range(dims)])
                term4 = sum([Gamma[rho][nu][lam] * Gamma[lam][mu][sigma] for lam in range(dims)])
                Riemann[rho][sigma][mu][nu] = term1 - term2 + term3 - term4

end_sp = time.time()
sympy_time = end_sp - start_sp

print(f"Pure SymPy Execution Time: {sympy_time:.4f} seconds")

--- Running Pure SymPy Benchmark ---
Pure SymPy Execution Time: 0.2043 seconds


## Experiment 2: EinsteinEngine (SymEngine C++ Backend) 

Here we utilize our custom object-oriented architecture. Under the hood, the engine bridges seamlessly between Python and C++, caching operations and flattening loops to maximize CPU efficiency.

In [3]:
print("--- Running EinsteinEngine Benchmark ---")

start_ee = time.time()

# The entire pipeline in just two lines of clean OOP code!
# We set verbose=True to observe the internal pipeline routing.
metric_sch = MetricTensor(g_schwarzschild, syms, name="Schwarzschild", verbose=True)
riemann_sch = RiemannCurvatureTensor.from_metric(metric_sch, verbose=True)

end_ee = time.time()
ee_time = end_ee - start_ee

print(f"\nEinsteinEngine Execution Time: {ee_time:.4f} seconds")

# Calculate the speed multiplier
if ee_time > 0:
    multiplier = sympy_time / ee_time
    print(f"PERFORMANCE MULTIPLIER: {multiplier:.2f}x faster!")

--- Running EinsteinEngine Benchmark ---
[Schwarzschild] Instantiated in 4D spacetime.
[Schwarzschild] Tensor initialized with index configuration: 'll'.
[Schwarzschild] Métrica validada y lista para cálculos.
Triggering pipeline execution for metric: 'Schwarzschild'
Computing Christoffel Symbols for 'Schwarzschild' in C++...
[Christoffel_Schwarzschild] Instantiated in 4D spacetime.
[Christoffel_Schwarzschild] Connection initialized. Note: This object is NOT a tensor.
Building Riemann Curvature Tensor...
[RiemannCurvature] Instantiated in 4D spacetime.
[RiemannCurvature] Tensor initialized with index configuration: 'ulll'.

EinsteinEngine Execution Time: 0.0461 seconds
PERFORMANCE MULTIPLIER: 4.43x faster!


## Mathematical Validation 

Finally, we extract a known non-zero component of the Schwarzschild Riemann tensor ($R^r_{\ t r t}$) from both engines, apply an algebraic simplification, and verify that they yield the exact same physical result.

In [4]:
print("--- Cross-Engine Validation ---")

# 1. Extract results
raw_sympy = Riemann[1][0][1][0]
raw_ee = riemann_sch.get_component(1, 0, 1, 0)

# 2. Neutralize: Convert both to standard SymPy expressions
# This forces SymEngine results to become compatible with SymPy
comp_sympy = sp.simplify(raw_sympy)
comp_ee = sp.simplify(sp.sympify(raw_ee)) # <--- El truco está aquí: sp.sympify()

print(f"SymPy Result: {comp_sympy}")
print(f"EinsteinEngine Result: {comp_ee}")

# 3. Final validation using the string representation or difference
if str(comp_sympy) == str(comp_ee):
    print("\n SUCCESS: Both engines computed the exact same physical result.")
else:
    print(f"\n ERROR: Discrepancy detected.")

--- Cross-Engine Validation ---
SymPy Result: 2*M*(2*M - r)/r**4
EinsteinEngine Result: 2*M*(2*M - r)/r**4

 SUCCESS: Both engines computed the exact same physical result.


In [1]:
import time
import symengine as se
import sympy as sp

from einsteinengine.symbolic.metric import MetricTensor
from einsteinengine.symbolic.riemann import RiemannCurvatureTensor
from einsteinengine.symbolic.ricci_tensor import RicciTensor
from einsteinengine.symbolic.ricci_scalar import RicciScalar


# 1. Definimos las variables y FUNCIONES simbólicas arbitrarias
t, r, theta, phi = sp.symbols('t r theta phi')
syms = [t, r, theta, phi]

# Usamos funciones de SymPy que luego pasaremos a SymEngine
A = sp.Function('A')(r)
B = sp.Function('B')(r)

# 2. Construimos la Métrica General
g_general = [
    [-A, 0, 0, 0],
    [0,  B, 0, 0],
    [0,  0, r**2, 0],
    [0,  0, 0, r**2 * sp.sin(theta)**2]
]

print("--- Running EinsteinEngine Extreme Benchmark ---")
start_ee = time.time()

# 3. El Pipeline Completo (¡Mira qué limpio queda tu código de usuario!)
metric_gen = MetricTensor(g_general, syms, name="GeneralSpherical", verbose=False)
riemann_gen = RiemannCurvatureTensor.from_metric(metric_gen, verbose=False)
ricci_tensor_gen = RicciTensor.from_riemann(riemann_gen, verbose=False)
ricci_scalar_gen = RicciScalar.from_ricci_tensor_and_metric(ricci_tensor_gen, metric_gen, verbose=False)

end_ee = time.time()
ee_time = end_ee - start_ee

print(f"EinsteinEngine Execution Time: {ee_time:.4f} seconds")
print("\nEscalar de Ricci Calculado (Primeros 150 caracteres para no saturar la pantalla):")
print(str(ricci_scalar_gen.get_raw_data())[:150] + " ...")

--- Running EinsteinEngine Extreme Benchmark ---
EinsteinEngine Execution Time: 0.5221 seconds

Escalar de Ricci Calculado (Primeros 150 caracteres para no saturar la pantalla):
(-cos(theta)**2/sin(theta)**2 + (1/2)*r*Derivative(B(r), r)/B(r)**2 + (-1/2)*r*Derivative(A(r), r)/(B(r)*A(r)) - (-1 - cos(theta)**2/sin(theta)**2) -  ...
